# OpenMotion AI - MimicMotion SOTA Backend (Abril 2026)
Este notebook roda um motor de **Motion Transfer** de alta fidelidade, superior ao AnimateDiff, otimizado para a GPU T4 do Colab.

In [ ]:
# 1. SETUP AMBIENTE E DOWNLOAD DE PESOS
import os
import shutil
from huggingface_hub import snapshot_download

%cd /content
if os.path.exists('MimicMotion'):
    shutil.rmtree('MimicMotion')

!pip install -q fastapi uvicorn python-multipart xformers==0.0.25
!pip install -q diffusers transformers accelerate decord einops omegaconf
!pip install -q onnxruntime-gpu insightface

!git clone https://github.com/Tencent/MimicMotion.git
%cd /content/MimicMotion
!mkdir -p models

snapshot_download(repo_id="Tencent/MimicMotion", local_dir="models", allow_patterns=["*.pth", "*.bin", "*.json"])
print("✅ Tudo pronto!")

In [ ]:
%%writefile main.py
import uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
import torch
import os
import uuid
import shutil
import cv2
from mimic_motion_pipeline import MimicMotionPipeline

app = FastAPI()
pipe = MimicMotionPipeline.from_pretrained("models", torch_dtype=torch.float16)
pipe.to("cuda")
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

def save_video(frames, output_path, fps=15):
    height, width, _ = frames[0].shape
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    for frame in frames:
        out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    out.release()

@app.post("/generate")
async def generate(character_image: UploadFile = File(...), reference_video: UploadFile = File(...)):
    session_id = str(uuid.uuid4())
    work_dir = f"temp/{session_id}"
    os.makedirs(work_dir, exist_ok=True)
    img_path, vid_path, out_path = f"{work_dir}/i.png", f"{work_dir}/v.mp4", f"{work_dir}/o.mp4"

    with open(img_path, "wb") as f: shutil.copyfileobj(character_image.file, f)
    with open(vid_path, "wb") as f: shutil.copyfileobj(reference_video.file, f)

    result = pipe(ref_image=img_path, motion_video=vid_path, num_inference_steps=25, guidance_scale=4.0, fps=15, chunk_size=8)
    save_video(result.frames[0], out_path)
    return FileResponse(out_path, media_type="video/mp4")

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
# 3. INICIALIZAÇÃO DO TÚNEL E SERVER
print("Senha para o Localtunnel:")
!curl ipv4.icanhazip.com
!npm install -g localtunnel

import threading
def run_lt():
    !lt --port 8000
threading.Thread(target=run_lt, daemon=True).start()

!python main.py